In [53]:
import pandas as pd

In [54]:
cars = pd.read_csv("../data/Car Sales Missing Data.csv")

### data inspection

In [55]:
cars.shape  # (γραμμές, στήλες)

(1000, 5)

In [56]:
cars.head(5)

,Make,Colour,Odometer (KM),Doors,Price
0,Nissan,Black,10148.0,4.0,Rs1366850
1,Toyota,White,10217.0,4.0,Rs1144150
2,Toyota,White,10247.0,4.0,Rs1628300
3,Honda,White,10953.0,4.0,Rs831800
4,Nissan,Blue,10954.0,4.0,Rs1521950


In [57]:
cars = cars.rename(columns={'Make':'make',
                     'Colour':'colour',
                     'Odometer (KM)':'odometer (KM)',
                     'Doors':'doors',
                     'Price':'price'})

In [58]:
cars.info()  # βλέπω πόσες γραμμές, στήλες και NaN έχει

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   make           951 non-null    str    
 1   colour         950 non-null    str    
 2   odometer (KM)  950 non-null    float64
 3   doors          950 non-null    float64
 4   price          950 non-null    str    
dtypes: float64(2), str(3)
memory usage: 39.2 KB


παρατηρούμε από το info() ότι το Dtype της στήλης price είναι περασμένο σαν str.  
έχει τον χαρακτήρα Rs μπροστά από κάθε τιμή (ρούβλια)

In [59]:
cars['price'].head(3)
# θα πρέπει να αλλάξει μετά

0    Rs1366850
1    Rs1144150
2    Rs1628300
Name: price, dtype: str

In [60]:
cars.nunique()  # πόσες μοναδικές τιμές έχει κάθε στήλη

make               4
colour             5
odometer (KM)    948
doors              3
price            934
dtype: int64

In [61]:
cars['make'].value_counts(dropna=False) # πόσες φορές εμφανίζονται οι μοναδικές τιμές

make
Toyota    379
Honda     292
Nissan    183
BMW        97
NaN        49
Name: count, dtype: int64

In [62]:
cars['colour'].value_counts(dropna=False)

colour
White    390
Blue     302
Black     95
Red       88
Green     75
NaN       50
Name: count, dtype: int64

In [63]:
cars['doors'].value_counts(dropna=False) 
# domain knowledge & ερμηνεία δεδομένων: το πορτπαγκάζ πιάνεται σαν πόρτα σε μερικά 
# αυτοκίνητα αν ανοίγει από το πλάι. για αυτό οι αριθμοί 3, 5

doors
4.0    811
5.0     75
3.0     64
NaN     50
Name: count, dtype: int64

In [64]:
cars.duplicated().sum()  # υπάρχει μία διπλότυπη εγγραφή

np.int64(1)

In [65]:
cars[cars.duplicated(keep=False)] # βλέπω την διπλή εγγραφή

,make,colour,odometer (KM),doors,price
998,Toyota,White,NaN,4.0,NaN
999,Toyota,White,NaN,4.0,NaN


In [66]:
cars= cars.drop_duplicates()  #την σβήνω

### data cleaning

In [67]:
cars[
    (cars['make'].isna())|
    (cars['price'].isna())
].head(5)  
# αποφασίζω πως αν λείπει η μάρκα ή η τιμή θα σβήσω την γραμμή, μιας που είναι 
# αναντικατάστατες πληροφορίες

,make,colour,odometer (KM),doors,price
11,NaN,Black,13266.0,4.0,Rs725100
19,Toyota,Blue,17609.0,4.0,NaN
23,BMW,White,18394.0,5.0,NaN
29,NaN,White,19482.0,3.0,Rs1560400
40,NaN,Blue,23545.0,4.0,Rs696600


In [68]:
cars = cars.dropna(subset=['make','price'])  
#η subset δέχεται μια λίστα από τιμές, θα μπορούσα να το έκανα σε ξεχωριστά βήματα
#για κάθε στήλη εννοείται.

In [69]:
# γεμίζω τις NaN τιμές για τις άλλες στλήλες

cars['colour']= cars['colour'].fillna('Missing')

cars['doors']= cars['doors'].fillna('Missing')

cars['odometer (KM)']= cars['odometer (KM)'].fillna(cars['odometer (KM)'].median())

In [70]:
cars.isna().sum()   #καμία στήλη δεν έχει κενά!

make             0
colour           0
odometer (KM)    0
doors            0
price            0
dtype: int64

In [71]:
cars['price'] = cars['price'].str.replace('Rs', '').astype(float)


In [72]:
cars['price'].head()

0    1366850.0
1    1144150.0
2    1628300.0
3     831800.0
4    1521950.0
Name: price, dtype: float64

In [73]:
cars['price'].describe().round()

count        903.0
mean      806437.0
std       428480.0
min       139800.0
25%       482425.0
50%       722100.0
75%      1044300.0
max      2622900.0
Name: price, dtype: float64

### Exploratory Data Analysis - EDA

Αφού έκανες το Data Cleaning (την «αγγαρεία»), τώρα ξεκινάει το διασκεδαστικό κομμάτι. Σαν Data Analyst, σκέφτεσαι σαν ιδιοκτήτης μάντρας αυτοκινήτων και κάνεις **ερωτήσεις για να πάρεις επιχειρηματικές αποφάσεις**.

In [74]:
# ποια μάρκα είναι πιο ακριβή κατά μέσο όρο?
cars.groupby('make')['price'].mean().round().sort_values(ascending=False).reset_index(name='mean')

,make,mean
0,BMW,1345122.0
1,Toyota,785763.0
2,Honda,730933.0
3,Nissan,683225.0


In [75]:
# Ποιος συνδυασμός μάρκας και χρώματος πουλάει περισσότερο;
cars[['make','colour']].value_counts().reset_index(name='combination_count').head(5)

,make,colour,combination_count
0,Toyota,White,135
1,Honda,White,106
2,Toyota,Blue,106
3,Honda,Blue,89
4,Nissan,White,71


In [76]:
cars.groupby(['make','colour']).size().sort_values(ascending=False).head(5)

make    colour
Toyota  White     135
Honda   White     106
Toyota  Blue      106
Honda   Blue       89
Nissan  White      71
dtype: int64

In [77]:
make_colour_pivot = pd.crosstab(cars['make'], cars['colour'], margins=True)

make_colour_pivot.sort_values('All', ascending=False)

colour,Black,Blue,Green,Missing,Red,White,All
make,,,,,,,
All,85,278,70,45,79,346,903
Toyota,37,106,30,14,32,135,354
Honda,27,89,18,21,20,106,281
Nissan,13,54,16,5,16,71,175
BMW,8,29,6,5,11,34,93


In [86]:
# Ποια είναι τα 5 πιο ακριβά αυτοκίνητα του πίνακα;
cars.sort_values('price', ascending=False).head(5)

,make,colour,odometer (KM),doors,price
16,BMW,White,16274.0,5.0,2622900.0
186,BMW,White,57905.0,3.0,2543400.0
274,BMW,White,79937.0,5.0,2434300.0
296,BMW,White,85739.0,5.0,2420950.0
42,BMW,White,23996.0,5.0,2411950.0


In [85]:
#Τα περισσότερα χιλιόμετρα ρίχνουν όντως την τιμή; (corr - Συσχέτιση)
cars[['odometer (KM)', 'price']].corr().round(2)


,odometer (KM),price
odometer (KM),1.00,-0.39
price,-0.39,1.00
